In [36]:
import os
import json
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage


load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

一个完整的Agent至少要包含两个关键的部分：
- 模型：是Agent的大脑，负责推理、分析，规划任务步骤
- 工具：是Agent的手脚，负责执行任务，与外界交互

In [25]:

from langchain.tools import tool
# @tool
# def get_weather(location: str) -> str:
#     """
#     Get the weather in a given location.
#     Args:
#         location: city name or coordinates
#     """
#     return f"Current weather in {location} is sunny"

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5


In [26]:
from pydantic import BaseModel,Field
from typing import Literal

# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


In [34]:

agent = create_agent(model="deepseek-chat",tools=[get_weather,square_root])

#"杭州今天天气如何?"
content = "16开平方是多少"
response = agent.invoke(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
)

for message in response['messages']:
    message.pretty_print()


================================ Human Message =================================

杭州接下来几天天气如何?
================================== Ai Message ==================================

好的，我来查询杭州的天气情况，包括当前天气和未来几天的预报。
Tool Calls:
  get_weather (call_00_48admgLow4dhqq3nF6zl7868)
 Call ID: call_00_48admgLow4dhqq3nF6zl7868
  Args:
    location: 杭州
    include_forecast: True
================================= Tool Message =================================
Name: get_weather

Current weather in 杭州: 22 degrees C
Next 5 days: Sunny
================================== Ai Message ==================================

杭州最近的天气情况如下：

### ☀️ 当前天气
- **温度**：22°C
- **天气状况**：晴朗

### 📅 未来5天预报
未来几天杭州将持续以**晴天**为主，非常适合出行和户外活动！

不过需要注意的是，预报显示较为简略，建议你：
- 白天和早晚温差可能较大，出门可以带件薄外套
- 如果想了解更精确的每日气温范围（最高/最低温），建议查看当地气象台的详细预报

祝你在杭州生活/旅行愉快！如果有其他问题，随时问我~ 😊


In [33]:
get_weather.invoke({"location": "杭州", "include_forecast": True})
square_root.invoke({"x": 567})

23.811761799581316

In [37]:
from langchain_tavily import TavilySearch

# 初始化工具，并设置参数，具体参数设置参考官网
tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [39]:
tool.invoke("杭州今天天气如何？")

{'query': '杭州今天天气如何？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://weathernew.pae.baidu.com/weathernew/pc?query=%E6%B5%99%E6%B1%9F%E6%9D%AD%E5%B7%9E%E5%A4%A9%E6%B0%94&srcid=4982',
   'title': '杭州 - 百度',
   'content': '今天. 23 ~ 30°C 多云. 北风3级. 优 · 29. 20 ~ 29°C 小雨. 东北风3级. 良 · 30. 19 ~ 29°C 晴. 东北风2级. 良 · 31. 19 ~ 32°C 晴. 东北风3级. 良.',
   'score': 0.9997154,
   'raw_content': None},
  {'url': 'https://www.aqi.in/weather/cn/china/zhejiang/hangzhou/hangzhou',
   'title': '实时Hangzhou天气状况：气温| 30天预报 - AQI.in',
   'content': "# Hangzhou Weather Conditions Current Temperature Level. Air may feel slightly humid. Live Weather Ranking: World Hottest Cities 2026. 星期六 (六月 6) : 气温 24°C，湿度 78%，Moderate or heavy rain shower 天气状况。. ## **China**'s 最热 Cities - 五月 2026. 最后更新： 28 May 2026, 01:10 PM. Hangzhou 今天 28 五月 2026 的当前气温是多少？. Hangzhou 的当前气温为 28°C，体感温度为 32°C。今日预报显示最高气温 30°C，最低气温 24°C，截至 09:00 PM 28 五月 2026，全天温差为 6 度。. 以下是截至 09:00 PM 28 五月 2026 Hangzhou 当前天气状

In [42]:
agent = create_agent(
    model="deepseek-chat",
    tools=[tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)
for message in response['messages']:
    message.pretty_print()
# 调用工具
for chunk in agent.stream(
    {"messages": [HumanMessage(content="北京接下来5天天气如何?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

================================ Human Message =================================

杭州接下来几天天气如何?
================================== Ai Message ==================================

好的，我来查询杭州的天气情况，包括当前天气和未来几天的预报。
Tool Calls:
  get_weather (call_00_48admgLow4dhqq3nF6zl7868)
 Call ID: call_00_48admgLow4dhqq3nF6zl7868
  Args:
    location: 杭州
    include_forecast: True
================================= Tool Message =================================
Name: get_weather

Current weather in 杭州: 22 degrees C
Next 5 days: Sunny
================================== Ai Message ==================================

杭州最近的天气情况如下：

### ☀️ 当前天气
- **温度**：22°C
- **天气状况**：晴朗

### 📅 未来5天预报
未来几天杭州将持续以**晴天**为主，非常适合出行和户外活动！

不过需要注意的是，预报显示较为简略，建议你：
- 白天和早晚温差可能较大，出门可以带件薄外套
- 如果想了解更精确的每日气温范围（最高/最低温），建议查看当地气象台的详细预报

祝你在杭州生活/旅行愉快！如果有其他问题，随时问我~ 😊
step: model
content: [{'type': 'text', 'text': '我来帮你查询北京接下来5天的天气预报。'}, {'type': 'tool_call', 'id': 'call_00_Q1lJsZlNOdXvO7UaQPiL5274', 'name': 'tavily_search', 'args': {'query': '北